### LiveCodeBench-v6 Diagnostic Dataset 정제 기록

In [1]:
from datasets import load_dataset

HF_CACHE = "/mnt/hdd/hf_cache"

dataset = load_dataset(
    "livecodebench/code_generation_lite",
    version_tag="release_v6",
    cache_dir=HF_CACHE,
)

lcb_v6 = dataset["test"]

print("=" * 60)
print("LiveCodeBench v6 Dataset")
print("=" * 60)

print(f"Number of samples : {len(lcb_v6)}")
print(f"Columns           : {lcb_v6.column_names}")

/mnt/hdd/conda_envs/slm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LiveCodeBench v6 Dataset
Number of samples : 1055
Columns           : ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata']


In [2]:
import json
from datasets import concatenate_datasets

# ============================================================
# 1. Evaluation type 확인 함수
# ============================================================

def get_test_types(sample):
    """public_test_cases에서 testtype 목록을 반환"""
    
    test_cases = sample["public_test_cases"]

    if isinstance(test_cases, str):
        test_cases = json.loads(test_cases)

    if not test_cases:
        return []

    return [test.get("testtype") for test in test_cases]


def get_evaluation_type(sample):
    """
    문제의 evaluation type을 반환.

    모든 public test case가 동일한 testtype을 가지는 경우만
    해당 type으로 분류한다.
    """
    
    test_types = get_test_types(sample)

    if not test_types:
        return None

    unique_types = set(test_types)

    if len(unique_types) == 1:
        return test_types[0]

    # 한 문제 안에 여러 evaluation type이 섞인 경우
    return None


# ============================================================
# 2. Platform 필터
# ============================================================

diagnostic_pool = lcb_v6.filter(
    lambda x: x["platform"] in ["atcoder", "leetcode"]
)

print("=" * 70)
print("Platform Filter")
print("=" * 70)
print(f"Original LCB v6 : {len(lcb_v6)}")
print(f"AtCoder/LeetCode: {len(diagnostic_pool)}")


# ============================================================
# 3. Evaluation type 필터
# ============================================================

evaluation_types = []

for sample in diagnostic_pool:
    evaluation_type = get_evaluation_type(sample)
    evaluation_types.append(evaluation_type)

print("\n" + "=" * 70)
print("Evaluation Type Distribution")
print("=" * 70)

from collections import Counter

eval_counter = Counter(evaluation_types)

for eval_type, count in eval_counter.items():
    print(f"{str(eval_type):<15}: {count}")


# ============================================================
# 4. Difficulty × Evaluation Type별 Sampling
# ============================================================

samples = []

for evaluation_type in ["stdin", "functional"]:

    for difficulty in ["easy", "medium", "hard"]:

        # 조건 필터
        subset = diagnostic_pool.filter(
            lambda x:
                x["difficulty"] == difficulty
                and get_evaluation_type(x) == evaluation_type
        )

        # 최신순 정렬
        subset = subset.sort(
            "contest_date",
            reverse=True
        )

        # 최대 100개
        n = min(100, len(subset))

        subset = subset.select(range(n))

        samples.append(subset)

        print(
            f"{evaluation_type:<12} "
            f"{difficulty:<8} "
            f"available={len(subset):>3} "
            f"selected={n:>3}"
        )


# ============================================================
# 5. 최종 Diagnostic Dataset
# ============================================================

lcb_diagnostic = concatenate_datasets(samples)


# ============================================================
# 6. 최종 결과
# ============================================================

print("\n" + "=" * 70)
print("Final LiveCodeBench Diagnostic Dataset")
print("=" * 70)

print(f"Total samples : {len(lcb_diagnostic)}")

print("\nEvaluation Type × Difficulty")

for evaluation_type in ["stdin", "functional"]:

    for difficulty in ["easy", "medium", "hard"]:

        count = sum(
            1
            for x in lcb_diagnostic
            if get_evaluation_type(x) == evaluation_type
            and x["difficulty"] == difficulty
        )

        print(
            f"{evaluation_type:<12} "
            f"{difficulty:<8}: {count}"
        )

Platform Filter
Original LCB v6 : 1055
AtCoder/LeetCode: 1046

Evaluation Type Distribution
functional     : 444
stdin          : 601
None           : 1
stdin        easy     available=100 selected=100
stdin        medium   available=100 selected=100
stdin        hard     available=100 selected=100
functional   easy     available=100 selected=100
functional   medium   available=100 selected=100
functional   hard     available=100 selected=100

Final LiveCodeBench Diagnostic Dataset
Total samples : 600

Evaluation Type × Difficulty
stdin        easy    : 100
stdin        medium  : 100
stdin        hard    : 100
functional   easy    : 100
functional   medium  : 100
functional   hard    : 100


In [3]:
lcb_diagnostic

Dataset({
    features: ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata'],
    num_rows: 600
})

In [ ]:
# from pathlib import Path
# # ============================================================
# # 1. 평가 방식별 분리
# # ============================================================

# lcb_stdin = lcb_diagnostic.filter(
#     lambda x: get_evaluation_type(x) == "stdin"
# )

# lcb_functional = lcb_diagnostic.filter(
#     lambda x: get_evaluation_type(x) == "functional"
# )


# # ============================================================
# # 2. 저장 경로
# # ============================================================

# output_dir = Path(
#     "/mnt/hdd/project_sLM_planning/data/livecodebench_v6"
# )

# output_dir.mkdir(parents=True, exist_ok=True)


# # ============================================================
# # 3. HuggingFace Dataset 저장
# # ============================================================

# lcb_stdin.save_to_disk(
#     str(output_dir / "stdin")
# )

# lcb_functional.save_to_disk(
#     str(output_dir / "functional")
# )

Saving the dataset (3/3 shards): 100%|██████████| 300/300 [00:00<00:00, 3065.86 examples/s] 


In [7]:
# ============================================================
# 4. 확인
# ============================================================

print("=" * 70)
print("LiveCodeBench-v6 Diagnostic Dataset Saved")
print("=" * 70)

print(f"stdin      : {len(lcb_stdin)}")
print(f"functional : {len(lcb_functional)}")
print(f"total      : {len(lcb_stdin) + len(lcb_functional)}")

print(f"\nSaved to:")
print(f"  stdin      -> {output_dir / 'stdin'}")
print(f"  functional -> {output_dir / 'functional'}")

LiveCodeBench-v6 Diagnostic Dataset Saved
stdin      : 300
functional : 300
total      : 600

Saved to:
  stdin      -> /mnt/hdd/project_sLM_planning/data/livecodebench_v6/stdin
  functional -> /mnt/hdd/project_sLM_planning/data/livecodebench_v6/functional


---
### 추가 분석

In [ ]:
from collections import Counter

platform_counter = Counter(
    sample["platform"]
    for sample in lcb_diagnostic
)

print("=" * 70)
print("Platform Distribution")
print("=" * 70)

print(f"Total samples : {len(lcb_diagnostic)}")
print()

for platform, count in platform_counter.most_common():
    print(
        f"{platform:<15} "
        f"{count:>4} "
        f"({count / len(lcb_diagnostic) * 100:>5.1f}%)"
    )

In [9]:
print("=" * 70)
print("Platform × Evaluation Type × Difficulty")
print("=" * 70)

for platform in ["atcoder", "leetcode"]:
    print(f"\n[{platform}]")

    for eval_type, dataset in [
        ("stdin", lcb_stdin),
        ("functional", lcb_functional)
    ]:
        subset = dataset.filter(
            lambda x: x["platform"] == platform
        )

        print(f"\n  {eval_type}")

        for difficulty in ["easy", "medium", "hard"]:
            count = sum(
                1
                for x in subset
                if x["difficulty"] == difficulty
            )

            print(f"    {difficulty:<8}: {count}")

Platform × Evaluation Type × Difficulty

[atcoder]

  stdin
    easy    : 100
    medium  : 100
    hard    : 100

  functional
    easy    : 0
    medium  : 0
    hard    : 0

[leetcode]

  stdin
    easy    : 0
    medium  : 0
    hard    : 0

  functional
    easy    : 100
    medium  : 100
    hard    : 100


In [10]:
from collections import Counter
import json

def get_evaluation_type(sample):
    test_cases = sample["public_test_cases"]

    if isinstance(test_cases, str):
        test_cases = json.loads(test_cases)

    if not test_cases:
        return None

    test_types = set(
        test["testtype"]
        for test in test_cases
    )

    if len(test_types) == 1:
        return next(iter(test_types))

    return None


counter = Counter()

for sample in lcb_diagnostic:
    platform = sample["platform"]
    eval_type = get_evaluation_type(sample)

    counter[(platform, eval_type)] += 1


print("=" * 70)
print("Platform × Evaluation Type")
print("=" * 70)

for (platform, eval_type), count in sorted(counter.items()):
    print(f"{platform:<15} {eval_type:<15} {count:>4}")

Platform × Evaluation Type
atcoder         stdin            300
leetcode        functional       300
